In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load Dataset 1
tickets = pd.read_csv('../data/raw/customer_support_tickets.csv')

# .shape tells us: (number of rows, number of columns)
print(f"Shape: {tickets.shape}")
print(f"Rows: {tickets.shape[0]:,}  |  Columns: {tickets.shape[1]}")

Shape: (8469, 17)
Rows: 8,469  |  Columns: 17


In [3]:
# These are the 'features' — each column is a piece of information
print("Columns:")
for i, col in enumerate(tickets.columns, 1):
    print(f"  {i}. {col}")

Columns:
  1. Ticket ID
  2. Customer Name
  3. Customer Email
  4. Customer Age
  5. Customer Gender
  6. Product Purchased
  7. Date of Purchase
  8. Ticket Type
  9. Ticket Subject
  10. Ticket Description
  11. Ticket Status
  12. Resolution
  13. Ticket Priority
  14. Ticket Channel
  15. First Response Time
  16. Time to Resolution
  17. Customer Satisfaction Rating


In [4]:
# .head(3) shows first 3 rows — like previewing a spreadsheet
tickets.head(3)

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0


In [5]:
# CONCEPT - Missing Values (NaN = Not a Number):
# Real data is messy. Some fields are empty/null.
# We need to know which columns have gaps before we use them.
# isnull() returns True/False per cell, .sum() counts the Trues per column

missing = tickets.isnull().sum()
missing_pct = (missing / len(tickets) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Missing Values Report:")
print(missing_report[missing_report['Missing Count'] > 0])

Missing Values Report:
                              Missing Count  Missing %
Customer Satisfaction Rating           5700      67.30
Resolution                             5700      67.30
Time to Resolution                     5700      67.30
First Response Time                    2819      33.29


In [6]:
# These are the columns WE care about for the knowledge base
key_cols = ['Ticket Description', 'Resolution', 'Ticket Type', 
            'Product Purchased', 'Ticket Priority', 'Ticket Status']

print("=== KEY COLUMNS ANALYSIS ===\n")
for col in key_cols:
    if col in tickets.columns:
        non_null = tickets[col].notna().sum()
        unique = tickets[col].nunique()
        print(f"📋 '{col}'")
        print(f"   Non-null: {non_null:,} | Unique values: {unique:,}")
        # Show sample values for categorical columns
        if unique < 15:
            print(f"   Values: {tickets[col].value_counts().to_dict()}")
        print()

=== KEY COLUMNS ANALYSIS ===

📋 'Ticket Description'
   Non-null: 8,469 | Unique values: 8,077

📋 'Resolution'
   Non-null: 2,769 | Unique values: 2,769

📋 'Ticket Type'
   Non-null: 8,469 | Unique values: 5
   Values: {'Refund request': 1752, 'Technical issue': 1747, 'Cancellation request': 1695, 'Product inquiry': 1641, 'Billing inquiry': 1634}

📋 'Product Purchased'
   Non-null: 8,469 | Unique values: 42

📋 'Ticket Priority'
   Non-null: 8,469 | Unique values: 4
   Values: {'Medium': 2192, 'Critical': 2129, 'High': 2085, 'Low': 2063}

📋 'Ticket Status'
   Non-null: 8,469 | Unique values: 3
   Values: {'Pending Customer Response': 2881, 'Open': 2819, 'Closed': 2769}



In [7]:
# CONCEPT - This is exactly what will go into the AI's knowledge base.
# Read this carefully — the AI will learn from thousands of these.

sample = tickets[tickets['Resolution'].notna()].iloc[5]

print("=" * 60)
print("SAMPLE KNOWLEDGE UNIT (What the AI will learn from)")
print("=" * 60)
print(f"\nPRODUCT  : {sample.get('Product Purchased', 'N/A')}")
print(f"ISSUE    : {sample.get('Ticket Type', 'N/A')}")
print(f"PRIORITY : {sample.get('Ticket Priority', 'N/A')}")
print(f"\nCUSTOMER SAID:\n{sample.get('Ticket Description', 'N/A')}")
print(f"\nAGENT RESOLVED IT WITH:\n{sample.get('Resolution', 'N/A')}")
print("=" * 60)

SAMPLE KNOWLEDGE UNIT (What the AI will learn from)

PRODUCT  : Sony PlayStation
ISSUE    : Billing inquiry
PRIORITY : High

CUSTOMER SAID:
I'm having an issue with the {product_purchased}. Please assist.

Product Name: TPUBASK3E3KQ0


Join Date: Oct 2007 Posts: 11,532

Quote: I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?

AGENT RESOLVED IT WITH:
Officer moment world sing parent available.


In [8]:
# Load Dataset 2
# Note: The filename might vary — adjust if needed
import os

# Find the actual filename
faq_files = [f for f in os.listdir('../data/raw') 
             if 'bitext' in f.lower() or 'train' in f.lower()]
print("FAQ-related files:", faq_files)

FAQ-related files: ['Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv']


In [9]:
# Load it (adjust filename based on what printed above)
faq = pd.read_csv('../data/raw/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')  # Bitext usually saves as train.csv

print(f"Shape: {faq.shape}")
print(f"\nColumns: {faq.columns.tolist()}")
faq.head(3)

Shape: (26872, 5)

Columns: ['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


In [10]:
# Explore the categories — these are the TYPES of customer questions
print("Question Categories:")
print(faq['category'].value_counts().to_string())

Question Categories:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950


In [11]:
# See a real Q&A pair
sample_faq = faq.iloc[10]
print(f"\nCATEGORY : {sample_faq.get('category', 'N/A')}")
print(f"INTENT   : {sample_faq.get('intent', 'N/A')}")
print(f"\nQUESTION :\n{sample_faq.get('instruction', 'N/A')}")
print(f"\nANSWER   :\n{sample_faq.get('response', 'N/A')}")


CATEGORY : ORDER
INTENT   : cancel_order

QUESTION :
i dont know what to do to cancel order {{Order Number}}

ANSWER   :
I've taken note that you're unsure of the cancellation process for your order with the order number {{Order Number}}. Don't worry, I'm here to help you! To cancel your order, please follow these steps:

1. Sign into Your Account: Access our platform by signing in to your {{Online Company Portal Info}}. 
2. Find Your Order: Once logged in, navigate to your '{{Online Order Interaction}}' or '{{Online Order Interaction}}' tab to locate the order with the order number {{Order Number}}.
3. Initiate Cancellation: Click on the order and look for the option labeled '{{Online Order Interaction}}'. Select this option to initiate the cancellation process.
4. Confirm Cancellation: The system might prompt you to confirm the cancellation. If so, kindly provide the necessary information to proceed.
5. Review Cancellation Details: After confirming, you will receive a cancellation c

In [12]:
# Find ecommerce file
ecomm_files = [f for f in os.listdir('../data/raw') 
               if 'ecomm' in f.lower() or 'intent' in f.lower() or 'dialog' in f.lower()]
print("Ecommerce-related files:", ecomm_files)

Ecommerce-related files: ['Ecommerce_FAQ_intents.json']


In [13]:
# Load it (adjust filename as needed)
ecomm = pd.read_csv('../data/raw/Ecommerce_FAQ_intents.json')  # typical filename

print(f"Shape: {ecomm.shape}")
print(f"Columns: {ecomm.columns.tolist()}")
ecomm.head(5)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 2


In [14]:
import json
import pandas as pd

with open('../data/raw/Ecommerce_FAQ_intents.json') as f:
    data = json.load(f)

rows = []

for intent in data['intents']:
    tag = intent['tag']
    for pattern in intent['patterns']:
        rows.append({
            'tag': tag,
            'question': pattern,
            'responses': " | ".join(intent['responses'])
        })

ecomm = pd.DataFrame(rows)

print(ecomm.head())
print(ecomm.shape)

              tag                      question  \
0  create_account  How can I create an account?   
1  create_account             How do I sign up?   
2  create_account            I want to register   
3  create_account     Can I make a new account?   
4  create_account               Sign up process   

                                           responses  
0  To create an account, click on the 'Sign Up' b...  
1  To create an account, click on the 'Sign Up' b...  
2  To create an account, click on the 'Sign Up' b...  
3  To create an account, click on the 'Sign Up' b...  
4  To create an account, click on the 'Sign Up' b...  
(422, 3)


In [15]:
print("=" * 55)
print("   SMARTSERVE AI — KNOWLEDGE BASE SUMMARY")
print("=" * 55)

# Dataset 1
valid_tickets = tickets[tickets['Resolution'].notna()]
print(f"\n📁 Dataset 1: Customer Support Tickets")
print(f"   Total Records  : {len(tickets):,}")
print(f"   Usable Records : {len(valid_tickets):,} (have resolutions)")
print(f"   Issue Types    : {tickets['Ticket Type'].nunique()} categories")
print(f"   Products Covered: {tickets['Product Purchased'].nunique()} products")

# Dataset 2
print(f"\n📁 Dataset 2: Bitext FAQ Dataset")
print(f"   Total Q&A Pairs : {len(faq):,}")
print(f"   Categories      : {faq['category'].nunique()}")
print(f"   Unique Intents  : {faq['intent'].nunique()}")

print(f"\n{'='*55}")
print(f"   TOTAL KNOWLEDGE UNITS FOR AI : {len(valid_tickets) + len(faq):,}")
print(f"{'='*55}")
print("\n✅ Ready for Module 4 — Data Cleaning & Processing")

   SMARTSERVE AI — KNOWLEDGE BASE SUMMARY

📁 Dataset 1: Customer Support Tickets
   Total Records  : 8,469
   Usable Records : 2,769 (have resolutions)
   Issue Types    : 5 categories
   Products Covered: 42 products

📁 Dataset 2: Bitext FAQ Dataset
   Total Q&A Pairs : 26,872
   Categories      : 11
   Unique Intents  : 27

   TOTAL KNOWLEDGE UNITS FOR AI : 29,641

✅ Ready for Module 4 — Data Cleaning & Processing
